# *****Part 3: Applications for the first three filters

### Note: for the applications results, change the keywords list by searching "change" in the code below...

## # Results File Names

### Change year and topic prefix as needed

In [1]:
search_year = '2024' ## published years of articles, could be single year '2023' or multiple years '2022-2024'. 
## Note: It is advised to search for one year at a time, with maximum of 3 years because there is a limit on the number of articles retreived from Elsvier.

topic_analysis = "DMR"  ## define topic prefix to use in files/figures names.

In [2]:
### change file names ONLY if required, otherwise just change the year_analysis for other years of the same topic at the begining of part 1

## 1. File to save the full texts of the interested topic resutled from the LDA analysis
filename_full_text = f"{topic_analysis}_{search_year}_full_papers_text_fourth_filter.xlsx"   ### "fourth_filter" is used in the results file of parts 3 and 4   

## 2. File to save the titles, DOIs, and publication date of the interested topic resutled from the LDA analysis
filename_titles_dois = f"{topic_analysis}_{search_year}_full_papers_text_fourth_filter_titles_dois_date.xlsx"   


### Applications File Names

### 3. File to save the first application of the approach (counts of chemcial entites in the abstract)
filename_compounds_count = f"{topic_analysis}_app1_compounds_counts.xlsx"  ## search year included in the sheet name

### 4. File to save the second application (looking for keywords in abstract)
filename_abstract_count = f"{topic_analysis}_app2_abstract_counts.xlsx"

### 5. File to save the third application (looking for words in title and keywords)
filename_title_keywords_count = f"{topic_analysis}_app3_title_keywords_counts.xlsx"

### 6. File to save the fourth application: characterization techniques (the word counts in each article into csv file)
filename_fulltext_count = f"{topic_analysis}_{search_year}_app4_characterization_count.csv"    

### 7. File to save the fourth application; words counts for predefined threshold value
filename_fulltext_threshold = f"{topic_analysis}_{search_year}_app4_characterization_count_threshold.csv"     

### 8. File to save the experimental sections (fifth application) for each article into csv file (based on output condition above)
filename_fulltext_exp_sections = f"{topic_analysis}_app5_exp_sections.xlsx" ### save dataframe into csv


In [3]:
### To increase the width of the jupyter notebook code window (not part of the code, just to change the display width)
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:80% ! important; }<style>")) ### change -- percentage according to your preference or screen size

In [4]:
import pandas as pd
import numpy as np
import re 
import json
import matplotlib.pyplot as plt
%matplotlib inline
import os.path
import requests
import xml.etree.ElementTree as ET

## # Drop Missing Abstract

### Note: Missing abstracts are dropped from the working dataframe, to ensure only journal articles are left (books, reports, ..etc are removed)

In [5]:
## file to read the resutls of part 1 and part 2
filename_filtered = f'{topic_analysis}_{search_year}_three_filters.csv' 

### read the resutls from parts 1 and 2
df_123 = pd.read_csv(filename_filtered, index_col=0)

# df_123.head() ### dataframe to work with
df_123.columns

Index(['Title', 'Pub_name', 'DOI', 'Pub_date', 'Link', 'Abstract', 'Keywords'], dtype='object')

In [6]:
df_123.shape

(135, 7)

In [7]:
df_123 = df_123[df_123.Abstract != '0'] ## delete articles (rows) where the abstract is missing.

In [8]:
df_123.shape ## check how many articles are dropped for missing abstract by comparing to previous df.shape

(135, 7)

# Application 1: # ChemDataExtractor (Analyzing the abstracts)

### To extract "Compounds" column, which contain all chemical entities included in the abstract (One Appilication on the use of the filters) 

In [9]:
# # #### chemdataextractor installation if not already installed (need to run only once)
# %pip install chemdataextractor

In [10]:
# conda update chemdataextractor

In [11]:
# library imported
from chemdataextractor import Document
import chemdataextractor

In [12]:
df_123['Compounds'] = 0 ## new column to save the found compounds in the abstracts

In [13]:
df_123 = df_123.reset_index(drop=True) #### to ensure the index is reset
# df_123.head()
df_123.columns

Index(['Title', 'Pub_name', 'DOI', 'Pub_date', 'Link', 'Abstract', 'Keywords',
       'Compounds'],
      dtype='object')

In [14]:
################## Passing a text

for i in range(df_123.shape[0]):   #### Iterate through all abstract to find the compounds
    doc = Document(df_123['Abstract'][i])   ## Reading Document/Text
    array = np.array(doc.cems)   ## Use the "cems" property on a document or any document element to get a list of the chemical entity
    listToStr = ', '.join([str(elem) for elem in array])
    df_123['Compounds'][i] = listToStr # listToStr2
    
############## Chemical entity mention
df_123['Compounds']

C:\Users\ismae\AppData\Local\Temp\ipykernel_18292\1029033341.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_123['Compounds'][i] = listToStr # listToStr2


0      methane, carbon, carbon dioxide, carbon, carbo...
1      CO₂, nickel, carbon, CO, H₂, carbon, carbon, c...
2      silica, Ni, CH4, nickel, silica, TEOS, silica,...
3      SrZr0·5Ni0·4Fe0·1O3, nickel, perovskite, perov...
4      La2O3, methane, methane, LaNiO3, VMT, CH4, CH4...
                             ...                        
130    Ni, zeolite, methane, carbon, nickel, Ni, zeol...
131    carbon, Co, Ni, Carbon, Ni3Mo, Zn, Mo, Mn, Ni3...
132    Mg, xMgO-Al2O3, Mg, Ni, Ni, MgO-Al2O3, CO2, CH...
133    Ni, carbon, CO2, Ni, carbon, CH, Co, Co, Co, c...
134    CO2, Ni, Al(NO3)3·9H2O, Al2O3-CT, Ni, Ni, Ni, ...
Name: Compounds, Length: 135, dtype: object

### Note: You may compare between the abstract and the extracted compounds, at least for one example.

In [15]:
df_123['Abstract'][4], df_123['Compounds'][4]

("Dry reforming of methane (DRM) enables the conversion of two greenhouse gases, CH4 and CO2, into syngas, offering remarkable environmental and economic benefits. The primary challenge in this process lies in maintaining the catalyst's high catalytic activity and stability. In this study, natural vermiculite (VMT) was employed to modify the layer carrier with a large specific surface area and suitable pore size. By combining the structural characteristics of perovskite, we synthesized the LaNiO3/VMT-SiO2 perovskite catalyst using the citric acid complex method for methane dry reforming reaction. The results demonstrate that at T = 750 °C and GHSV = 36,000 mL/(h·gcat), the catalyst exhibits excellent activity (achieving maximum conversions of 82% for CO2 and 90% for CH4) as well as stability (up to 24 h). Characterization techniques such as XRD and TEM revealed that the catalyst possesses exceptional catalytic activity and stability due to its horizontally distributed active component 

### The main result of application 1 is to see see how many articles has "nickel" (or any other interested element) in them

### you may add more to the "word_list" list to look for more chemcial compounds (change)

In [16]:
## add to below list the interested chemical entites you want to calculate the count for them
words_list = ['ni']  ## check how many times any of these chemical enitities are reported in the abstract  ### change as needed....
compound_df=pd.DataFrame(columns=words_list)  ## initialize the final dataframe
list_other_comps = []

for i in range(df_123.shape[0]):   #### Iterate through all articles to find the compounds
    
    x_dict = dict.fromkeys(words_list, []) ## initialize the counter dictionary for each row (article)
    for word_find in words_list:  ### add words and occurance into the dictionary
   
        count = len(re.findall(r"\b{}\b".format(word_find), df_123['Abstract'][i].lower(), flags=re.IGNORECASE)) ## By setting the flags parameter to re.IGNORECASE, you are telling interpreter to ignore the case while performing the search.       
        x_dict[word_find] = count  ## add the value of how many times the word (chemical entity) in each abstract
    
    
    #**(Optional) other_comps is to find coumpound with 'ni' in them, which is specific to our case, you may change it as needed
    
    ## this is to find the chemical entities that starts with specific letters (it was used here to show entities composed of Ni)
    other_comps = re.findall(r"\b{}[a-zA-Z0-9\-()]+".format('ni'), df_123['Abstract'][i].lower(), flags=re.IGNORECASE)      
    list_other_comps.append(other_comps)

    compound_df_init=pd.DataFrame(x_dict, index=[i])              ## counter dictionary into dataframe              
    compound_df = pd.concat([compound_df, compound_df_init], axis=0)  ### add rows to the final counter dataframe

df_123['Interested_compounds'] = list_other_comps  ## add the other compounds to the original dataframe
# print(compound_df)

### Save The results of App1 in file

In [17]:
df_123_test = pd.concat([df_123, compound_df], axis=1)  ### add compounds counts dataframe into the original dataframe
df_123_test.columns

Index(['Title', 'Pub_name', 'DOI', 'Pub_date', 'Link', 'Abstract', 'Keywords',
       'Compounds', 'Interested_compounds', 'ni'],
      dtype='object')

In [18]:
df_123_app1 = df_123_test.drop(['Title', 'Pub_name', 'Pub_date', 'Link', 'Abstract'], axis=1)  ### final results of application 1 (chemical entities)
df_123_app1.columns

Index(['DOI', 'Keywords', 'Compounds', 'Interested_compounds', 'ni'], dtype='object')

In [19]:
df_123_app1

,DOI,Keywords,Compounds,Interested_compounds,ni
0,10.1016/j.joei.2024.101870,"['Methane reforming', 'Hydrogen-rich reducing ...","methane, carbon, carbon dioxide, carbon, carbo...",[nickel-based],0
1,10.1016/j.ijhydene.2024.11.412,"['Methane dry reforming', 'Carbon deposition',...","CO₂, nickel, carbon, CO, H₂, carbon, carbon, c...",[nickel-based],0
2,10.1016/j.apsadv.2024.100663,"['Dry reforming of methane', 'CO2 utilization'...","silica, Ni, CH4, nickel, silica, TEOS, silica,...","[nickel, ni-impregnated, nickel]",4
3,10.1016/j.ijhydene.2024.10.119,"['Perovskite', 'Dry methane reforming', 'Citra...","SrZr0·5Ni0·4Fe0·1O3, nickel, perovskite, perov...",[nickel],0
4,10.1016/j.ijhydene.2024.07.413,"['Perovskite', 'Vermiculite', 'Methane dry ref...","La2O3, methane, methane, LaNiO3, VMT, CH4, CH4...",[],2
...,...,...,...,...,...
130,10.1016/j.fuel.2023.129548,"['Strategy', 'Resistance to sintering', 'Resis...","Ni, zeolite, methane, carbon, nickel, Ni, zeol...",[nickel],7
131,10.1016/j.apsusc.2023.158958,"['Dry reforming of methane', 'Ni3X alloys', 'A...","carbon, Co, Ni, Carbon, Ni3Mo, Zn, Mo, Mn, Ni3...","[ni-based, ni3x-type, ni-based, ni3x, ni3v, ni...",3
132,10.1016/j.fuel.2023.130029,"['Dry reforming of methane', 'Ni0.5/xMgO-Al2O3...","Mg, xMgO-Al2O3, Mg, Ni, Ni, MgO-Al2O3, CO2, CH...","[ni0, nial2o4, nixmgyo, ni0]",5
133,10.1016/j.seppur.2023.125868,"['DRM', 'Co segregation', 'Ni based catalyst',...","Ni, carbon, CO2, Ni, carbon, CH, Co, Co, Co, c...",[],6


### (Comment/Uncomment) -- App1 results

In [20]:
# if os.path.isfile(filename_compounds_count): ### check if file exist, then add sheet, otherwise make new file
#     with pd.ExcelWriter(filename_compounds_count, engine = 'openpyxl', mode='a') as writer_app1:
#         df_123_app1.to_excel(writer_app1, sheet_name = f"{search_year}", index=True)  ### save the resulted rows, each year in different sheet
# else:
#     with pd.ExcelWriter(filename_compounds_count, engine = 'openpyxl') as writer_app1:
#         df_123_app1.to_excel(writer_app1, sheet_name = f"{search_year}", index=True)  ### save the resulted rows, each year in different sheet
# #     writer_app1.close()

# Application 2: # word count in Abstract

### keyowrds: ML, Data, Machine Learning, model, ANN, Neural...... Find the frequency of these keywords in the abstract

## Note: These keywords are applied to the abstract after applying the previous keywords in parts 1 and 2, where we searched for ('ni', 'nickel') in the abstract..... therefore the resulted articles by applying the abstract filter twice....

In [21]:
df_123.head()

,Title,Pub_name,DOI,Pub_date,Link,Abstract,Keywords,Compounds,Interested_compounds
0,Thermodynamic and molecular dynamics study of ...,Journal of the Energy Institute,10.1016/j.joei.2024.101870,2024-12-31,https://api.elsevier.com/content/article/doi/1...,The carbon neutrality strategy presents both c...,"['Methane reforming', 'Hydrogen-rich reducing ...","methane, carbon, carbon dioxide, carbon, carbo...",[nickel-based]
1,Numerical analysis of spatial distribution of ...,International Journal of Hydrogen Energy,10.1016/j.ijhydene.2024.11.412,2024-12-02,https://api.elsevier.com/content/article/doi/1...,This study investigates carbon deposition duri...,"['Methane dry reforming', 'Carbon deposition',...","CO₂, nickel, carbon, CO, H₂, carbon, carbon, c...",[nickel-based]
2,Steering catalytic property and reactivity of ...,Applied Surface Science Advances,10.1016/j.apsadv.2024.100663,2024-12-31,https://api.elsevier.com/content/article/doi/1...,The dry reforming of methane (DRM) is a promis...,"['Dry reforming of methane', 'CO2 utilization'...","silica, Ni, CH4, nickel, silica, TEOS, silica,...","[nickel, ni-impregnated, nickel]"
3,Influence of perovskite catalysts synthesis me...,International Journal of Hydrogen Energy,10.1016/j.ijhydene.2024.10.119,2024-11-19,https://api.elsevier.com/content/article/doi/1...,Perovskites refer to ceramic materials with an...,"['Perovskite', 'Dry methane reforming', 'Citra...","SrZr0·5Ni0·4Fe0·1O3, nickel, perovskite, perov...",[nickel]
4,The methane dry reforming reaction was conduct...,International Journal of Hydrogen Energy,10.1016/j.ijhydene.2024.07.413,2024-09-11,https://api.elsevier.com/content/article/doi/1...,Dry reforming of methane (DRM) enables the con...,"['Perovskite', 'Vermiculite', 'Methane dry ref...","La2O3, methane, methane, LaNiO3, VMT, CH4, CH4...",[]


In [22]:
# df_123['Abstract']

In [23]:
def word_counter_dataframe(words_find, dataframe, column_name,doi, i, title):
    '''
    words_find : list of words to find their occurance
    dataframe : dataframe to search 
    column_name: which column to iterate through in the dataframe
    doi : DOI of the article paragraphs
    index: the original index (row number) of the article
    
    funciton output: the occurance of each word (words_find) in each article (row) as new column                                  
    '''
    counters_characterization = {'doi': doi, 'title': title} ### dictionary to save the occurance of each word (for particular journal article defined by the DOI)
### apply the word counter funciton for each row in chosen column (for example the abstract) (df_papers['sections'][i]) for every article (df_links['DOI'][j])
    if isinstance(dataframe[column_name][i], str):  ### check if string, not to include the nan values for sections that has no paragraphs

        for word_find in words_find:  ### add words and occurance into the dictionary
            count = len(re.findall(r"\b{}".format(word_find), dataframe[column_name][i], flags=re.IGNORECASE)) ## By setting the flags parameter to re.IGNORECASE, you are telling interpreter to ignore the case while performing the search.

            if word_find in counters_characterization: ### check if word already exist in the dictionary to update value, otherwise add the new word
                counters_characterization[word_find] += count
            else:
                counters_characterization[word_find] = count       
    
    return pd.DataFrame([counters_characterization])  ## convert to dataframe

### Change keyowrds below as needed (change)

In [24]:
outputs_final_ML = pd.DataFrame()  ## new dataframe to save the results from ALL articles

### define list of words to search for in the article ## change -- this list to be updated by the user according to their requirements
words_find_outputs_ML = ['ML', 'Data', 'Machine', 'Learning', 'model', 'ANN', 'Neural'] ### change as needed....
    
for i in np.arange(df_123.shape[0]):  ## loop for the columns size

### apply the word counter funciton for each row in the column 'Abstract'
    outputs_ML = word_counter_dataframe(words_find_outputs_ML,df_123, 'Abstract', df_123['DOI'][i], i, df_123['Title'][i])  #### returns a dataframe
    
#### append article dataframe into the final results
    outputs_final_ML = outputs_final_ML._append(outputs_ML, ignore_index = True)

In [25]:
display(outputs_final_ML)

,doi,title,ML,Data,Machine,Learning,model,ANN,Neural
0,10.1016/j.joei.2024.101870,Thermodynamic and molecular dynamics study of ...,0,0,0,0,0,0,0
1,10.1016/j.ijhydene.2024.11.412,Numerical analysis of spatial distribution of ...,0,0,0,0,0,0,0
2,10.1016/j.apsadv.2024.100663,Steering catalytic property and reactivity of ...,0,0,0,0,0,0,0
3,10.1016/j.ijhydene.2024.10.119,Influence of perovskite catalysts synthesis me...,0,0,0,0,0,0,0
4,10.1016/j.ijhydene.2024.07.413,The methane dry reforming reaction was conduct...,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
130,10.1016/j.fuel.2023.129548,A strategy of constructing the Ni@silicalite-1...,1,0,0,0,0,0,0
131,10.1016/j.apsusc.2023.158958,Ni<ce:inf loc=post>3</ce:inf>X-type alloy cata...,0,0,0,0,1,0,0
132,10.1016/j.fuel.2023.130029,Flower-like hollow Ni<ce:inf loc=post>0.5</ce:...,1,0,0,0,0,0,0
133,10.1016/j.seppur.2023.125868,Understanding the role of Co segregation on th...,0,0,0,0,0,0,0


In [26]:
### remove rows with all column values is equal to zero

outputs_final_ML_condition = outputs_final_ML.drop(['doi', 'title'], axis=1)  ### from the 'doi' column to apply integer conditions
outputs_final_ML_condition = outputs_final_ML_condition.loc[(outputs_final_ML_condition!=0).any(axis=1)] ## remove rows with all column values is equal to zero

In [27]:
### Final results

index_outputs_final_ML = list(outputs_final_ML_condition.index) ## save the indices of the interested rows in a list
outputs_final_ML.iloc[index_outputs_final_ML] ## apply the indices list into the original dataframe to deteremine the DOIs of the interested articles

,doi,title,ML,Data,Machine,Learning,model,ANN,Neural
4,10.1016/j.ijhydene.2024.07.413,The methane dry reforming reaction was conduct...,1,0,0,0,0,0,0
10,10.1016/j.ceja.2024.100655,Enhanced syngas production through dry reformi...,0,1,0,0,3,0,0
17,10.1016/j.jece.2024.112308,Green technology for carbon dioxide utilizatio...,1,0,0,0,0,0,0
22,10.1016/j.apcatb.2024.123891,Unlocking exceptional diffusion of porous Ni@S...,1,0,0,0,0,0,0
23,10.1016/j.mtchem.2024.102255,Molybdate-based double perovskite materials in...,0,0,0,0,1,0,0
27,10.1016/j.mcat.2024.114216,Enhanced predictive optimization of methane dr...,0,0,0,0,2,3,2
30,10.1016/j.jre.2024.07.030,DFT-based activity and stability analysis of d...,0,0,0,0,1,0,0
33,10.1016/j.cep.2024.109899,Syngas production via microwave-assisted Dry R...,2,0,0,0,0,0,0
39,10.1016/j.ijhydene.2024.04.365,Influence of nickel contents on synthesis gas ...,1,0,0,0,0,0,0
41,10.1016/j.cej.2024.154193,Influence of catalyst shape on plasma-assisted...,0,1,0,0,0,0,0


### (Comment/Uncomment)  -- App2 results

In [28]:
if os.path.isfile(filename_abstract_count): ### check if file exist, then add sheet, otherwise make new file
    with pd.ExcelWriter(filename_abstract_count, engine = 'openpyxl', mode='a') as writer_app2:
        outputs_final_ML.iloc[index_outputs_final_ML].to_excel(writer_app2, sheet_name = f"{search_year}", index=True)  ### save the resulted rows, each year in different sheet
else:
    with pd.ExcelWriter(filename_abstract_count, engine = 'openpyxl') as writer_app2:
        outputs_final_ML.iloc[index_outputs_final_ML].to_excel(writer_app2, sheet_name = f"{search_year}", index=True)  ### save the resulted rows, each year in different sheet
#     writer_app3.close()

# Application 3: word count in title and keywords:

### change the keywords list below as needed. (change)

In [ ]:
words_find_title_keywords = ['throughput', 'influence', 'promoter', 'screening'] ## words to look for in the titles and the keywords of the articles  ### change as needed....

In [ ]:
## initiate lists to save the resulted dataframes
index_title = []  ## list for title match
index_keywords = [] ## list for keywords match

In [ ]:
for word in words_find_title_keywords: ### repeat for all words
    display(df_123[df_123['Title'].str.contains(word)]) ## display the result of the title match    
    index_title = index_title + list(df_123[df_123['Title'].str.contains(word)].index)
    print("--------")
    display(df_123[df_123['Keywords'].str.contains(word)]) ## display the result of the keywords match   
    index_keywords = index_keywords + list(df_123[df_123['Keywords'].str.contains(word)].index)
    print("Next word....")

### Save the results of all keywords

In [ ]:
index_title_keywords = index_title + index_keywords ## combine lists from both title match and keywords match
index_title_keywords = list(set(index_title_keywords)) ## remove duplciates
df_123_app3 = df_123.drop(['Pub_name', 'Pub_date', 'Link', 'Abstract', 'Compounds', 'Interested_compounds'], axis=1)  ### final results of application 3 (chemical entities)
df_123_app3.iloc[index_title_keywords] ## apply the indices list into the original dataframe to deteremine the DOIs of the interested articles

### (Comment/Uncomment)  -- App3 results

In [ ]:
if os.path.isfile(filename_title_keywords_count): ### check if file exist, then add sheet, otherwise make new file
    with pd.ExcelWriter(filename_title_keywords_count, engine = 'openpyxl', mode='a') as writer_app3:
        df_123_app3.iloc[index_title_keywords].to_excel(writer_app3, sheet_name = f"{search_year}", index=True)  ### save the resulted rows, each year in different sheet
else:
    with pd.ExcelWriter(filename_title_keywords_count, engine = 'openpyxl') as writer_app3:
        df_123_app3.iloc[index_title_keywords].to_excel(writer_app3, sheet_name = f"{search_year}", index=True)  ### save the resulted rows, each year in different sheet
        
#     writer_app3.close()

# *****Part 4: Apply the fourth filter (full text)

## (Comment/Uncomment) -- Full text is extracted after applying the previous three filters (title, keywords, and abstract) 

# # Extract Full Texts

### The results include two files: 
1. the sections of each artilce (each in seperate sheet)
2. file contains the artilces: titles, DOIs, and publication dates

In [ ]:
## Load configuration
con_file = open("config.json")
config = json.load(con_file)
con_file.close()

In [ ]:
#defining a header's dictionary to pass through requests
headers_dict = {"X-ELS-APIKey": config['apikey'], "X-ELS-Insttoken": config['insttoken'], "Accept": "application/xml"}
df_temp = pd.DataFrame()

######### File to save all the data, each paper in separate sheet #########
writer = pd.ExcelWriter(filename_full_text, engine = 'xlsxwriter')   

index_for_titles = []
paper_titles_list = []   ##### list to save the titles of the interested papers
doi_list = []  ##### list to save the dois of the interested papers
pub_date_list = []  ##### list to save the published date of the interested papers

    
for i in np.arange(0,df_123.shape[0],1):  #### loop through all links of the file generated from part 1 (only title filtered)    
    index_for_titles.append(i)
    paper_titles_list.append(df_123['Title'][i])
    doi_list.append(df_123['DOI'][i])
    pub_date_list.append(df_123['Pub_date'][i])

    #lists for dataframe
    label_list = [] 
    title_list = []
    para_list = []

    print(i, 'done' )
    #x takes response of the HTTP request, passes link
    x = requests.get(df_123['Link'][i], headers=headers_dict)

    #saving it as XML file
    with open("full_text.xml", 'wb') as f:
        f.write(x.content)

    #Reading the data
    #<?xml version="1.0" encoding="UTF-8"?>
    tree = ET.parse(r'full_text.xml')
    root = tree.getroot()
    original_text = root.find('{http://www.elsevier.com/xml/svapi/article/dtd}originalText')

    if original_text != None:
        doc = original_text.find('{http://www.elsevier.com/xml/xocs/dtd}doc')
        serial_item = doc.find('{http://www.elsevier.com/xml/xocs/dtd}serial-item')

        if serial_item != None:
            article = serial_item.find('{http://www.elsevier.com/xml/ja/dtd}article')

            if article != None:
                body = article.find('{http://www.elsevier.com/xml/ja/dtd}body')

                if body != None:
                    sections = body.find('{http://www.elsevier.com/xml/common/dtd}sections')

                    for section1 in sections.findall('{http://www.elsevier.com/xml/common/dtd}section'): ### for main sections
                        if section1.find('{http://www.elsevier.com/xml/common/dtd}label') != None or section1.find('{http://www.elsevier.com/xml/common/dtd}section-title') != None: ########  added the second condition for the if statement, because some papers have no label for the sections
    ### Note: it seems that the label does not always exist (returns none, specially if some papers decided not to use indexing for the titles and subtitles, therefore the second condition for the if statement
                        #if section1 != None:
                            if section1.find('{http://www.elsevier.com/xml/common/dtd}label') != None: ########  added this condition, because some papers have no label for the sections
                                label = section1.find('{http://www.elsevier.com/xml/common/dtd}label').text
                                label_list.append(label)
                            section_title_list = []
                            section_title = section1.find('{http://www.elsevier.com/xml/common/dtd}section-title')
                            for sec in section_title.itertext():
                                section_title_list.append(sec)

                            section_title_list = "".join(section_title_list)
                            section_title_list = list(section_title_list.split()) 
                            section_title_list = " ".join(section_title_list)
                            title_list.append(section_title_list)
                            print(section_title_list)

                            total_para = []
                            for paragraph in section1.findall('{http://www.elsevier.com/xml/common/dtd}para'): ### for paragraphs after main section title (some times they are empty, or very short since sub-title contains usually contain the paragraphs)
                                if paragraph != None:
                                    paragraph1 = []
                                    for p in paragraph.itertext():   #### the section paragraph is divided among various <para> in the xml, therefore we have to find them all then join them together
                                        paragraph1.append(p)

                                    paragraph1 = "".join(paragraph1)
                                    paragraph1 = list(paragraph1.split()) 
                                    paragraph1 = " ".join(paragraph1)
                                    total_para.append(paragraph1)

                            total_para = " ".join(total_para)
                            #print(total_para)
                            para_list.append(total_para)
                            print('\n')

                            for section2 in section1.findall('{http://www.elsevier.com/xml/common/dtd}section'): ### for sub-sections
                                if section2.find('{http://www.elsevier.com/xml/common/dtd}label') !=None or section2.find('{http://www.elsevier.com/xml/common/dtd}section-title') != None:  ########  added the second condition for the if statement, because some papers have no label for the sections

                                    if section2.find('{http://www.elsevier.com/xml/common/dtd}label') != None:   ########  added this condition, because some papers have no label for the sub-sections
                                        label = section2.find('{http://www.elsevier.com/xml/common/dtd}label').text
                                        label_list.append(label)

                                        print('sub-section ', label)

                                    sub_section_title_list = []

                                    sub_section_title = section2.find('{http://www.elsevier.com/xml/common/dtd}section-title')
                                    for sub in sub_section_title.itertext():
                                        sub_section_title_list.append(sub)

                                    sub_section_title_list = "".join(sub_section_title_list)
                                    sub_section_title_list = list(sub_section_title_list.split()) 
                                    sub_section_title_list = " ".join(sub_section_title_list)
                                    title_list.append(sub_section_title_list)

                                    para1 = []
                                    for para in section2.itertext():        ### for paragraphs after sub-section title
                                        para1.append(para)
                                    para1 = "".join(para1)
                                    para1 = list(para1.split()) 
                                    para1 = " ".join(para1)
                                    #print(para1) 
                                    para_list.append(para1)
                                    print('\n')

    temp = dict(doi = np.array(df_123['DOI'][i]),    
         title = np.array(df_123['Title'][i]), 
         pub_name = np.array(df_123['Pub_name'][i] ),
         title_label = np.array(label_list),
         titles = np.array(title_list),
         text = np.array(para_list))

    ### the below print statements are to check the number of sections for each article, and if the type command resulted in None type, there is either no text (for example it is a book not an article) or something is wrong with the above analysis
    print(len(temp['text']), type(temp['text']))       
    print(len(temp['titles']), type(temp['titles']))

    df_temp = pd.DataFrame({'sections' : temp['titles'].tolist(), 'Text': temp['text'].tolist()}) 
    ### Note: This data frame (dt_temp) includes (sections name and its paragraphs), you may include more information (for example, authors, keywrods, ..etc) from above dataframe (temp)
    df_temp.head()

    ## Save each data frame into one excel file, each paper in seperate tab (sheet)     
    df_temp.to_excel(writer, sheet_name = f'x{i}', index=True)   ######## comment -- if already saved ....... 

# Save the papers titles in seperate file       
df_papers_titles = pd.DataFrame({'#': index_for_titles,'titles':paper_titles_list,'dois':doi_list,'pub_date':pub_date_list})    ### save title into dataframe
df_papers_titles.to_excel(filename_titles_dois, sheet_name = 'titles and dois and date', index=True)  ######## comment -- if already saved .....

writer.close()

In [ ]:
### To increase the width of the jupyter notebook code window (not part of the code, just to change the display width)
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% ! important; }<style>")) ### change -- percentage according to your preference or screen size

# Full Text Applications

# # Application 4: Find the characterization techniques performed in each study

### count the words in the full text....

### Read results file (full texts and titles, DOIs, dates)

In [ ]:
df_papers_titles = pd.read_excel(filename_titles_dois, index_col=0) ## read the titles, dois, and dates

### Function to count defined words in the article

In [ ]:
def word_counter_fulltext(words_find, df_papers,doi,index):
    '''
    words_find : list of words to find their occurance
    para_text : the paragraph text to search
    doi : DOI of the article paragraphs
    
    loopint through all sections (except the introduction) for a particular article, the function return a dataframe with the DOI and 
    the index, and the occurance of each word (in words_find) as columns                                  
    '''
    counters_fulltext = {'original_index':index, 'doi': doi} ### dictionary to save the occurance of each word (for particular journal article defined by the DOI)
    for i in np.arange(1, df_papers.shape[0]-1):  ## begin at 1: to exclude the introduction && -1 is: to exclude the last paragraph, usually the conclusion
    
    ### apply the word counter funciton for each section (df_papers['sections'][i]) for every article (df_links['DOI'][j])
        if isinstance(df_papers['Text'][i], str):  ### check if string, not to include the nan values for sections that has no paragraphs
            
            for word_find in words_find:  ### add words and occurance into the dictionary
                count = len(re.findall(r"\b{}\b".format(word_find), df_papers['Text'][i], flags=re.IGNORECASE)) ## By setting the flags parameter to re.IGNORECASE, you are telling interpreter to ignore the case while performing the search.
                if word_find in counters_fulltext: ### check if word already exist in the dictionary to update value, otherwise add the new word
                    counters_fulltext[word_find] += count
                else:
                    counters_fulltext[word_find] = count       
    
    return pd.DataFrame([counters_fulltext])  ## convert to dataframe

### Define the words to look for below.... (change)

###  (comment/uncomment): may take a long time, depending on the number of articles.... might want to save it so you do not run it again

In [ ]:
## comment if saved ...
characterization_count_final = pd.DataFrame()  ## new dataframe to save the results from ALL articles
    
all_sheets = pd.read_excel(filename_full_text, sheet_name=None) ## read the full texts to use the sheets names to iterate over the articles, as below,  
all_sheets.keys() ## sheet titles for iterations

### define list of words to search for in the article ## change -- this list to be updated by the user according to their requirements
words_find_characterization = ['TPR', 'BET', 'TPD','ICP', 'TG|TGA', 'TPO'] ### change as needed....

for i, j in enumerate(all_sheets.keys()): ## go through the sheets of the articles that we extracted the full texts 

    ### read (has to be in the loop) the file to lead each article seperately, since they are in different sheets
    df_papers = pd.read_excel(filename_full_text, index_col=0, sheet_name = j)

    ### apply the word counter funciton for each section (df_papers['sections'][i]) for every article (df_links['DOI'][j])
    characterization_count = word_counter_fulltext(words_find_characterization,df_papers, df_papers_titles.iloc[i]['dois'], int(j[1:]))  #### returns a dataframe
    
    ### append article dataframe into the final results
    characterization_count_final = characterization_count_final._append(characterization_count, ignore_index = True)
    
### reset the index to match the doi of each article (each row represent the material characterization techniques used in each article) 
characterization_count_final.set_index('original_index', drop=True, inplace=True)  ## reset index as doi

In [ ]:
### Print characterization techniques DataFrame
pd.options.display.max_rows = None
display(characterization_count_final)

In [ ]:
# ### save the output words counts in each article into csv file
characterization_count_final.to_csv(filename_fulltext_count, index=True)  ## comment if already saved 

# # Analysis on the output results (filtering the articles) with predifined conditions

In [ ]:
# characterization_count_final = pd.read_csv(filename_fulltext_count, index_col=0) ## Read if not previously defined
characterization_count_final.head()

### Define suitable number of occurance (threshold value) to classify into True/False of the characterization technique

### Note: this value may depends on the topic searched, therefore, it is recommended to tune this value for each case.

In [ ]:
characterization_condition = characterization_count_final.drop('doi', axis=1) ## remove doi column to apply the conditions

## Classification (True or False) of characterization technique occurrnece baesd on various threshold values

df1 = characterization_condition.applymap(lambda x: True if x >=1 else False)
# df1.apply(pd.value_counts)

df2 = characterization_condition.applymap(lambda x: True if x >=2 else False)
# df2.apply(pd.value_counts)

df3 = characterization_condition.applymap(lambda x: True if x >=3 else False)
# df3.apply(pd.value_counts)

df4 = characterization_condition.applymap(lambda x: True if x >=4 else False)
# df4.apply(pd.value_counts)

df5 = characterization_condition.applymap(lambda x: True if x >=5 else False)
# df5.apply(pd.value_counts)

## Plot the results vs. the threshold value

In [ ]:
plt.rcParams.update({'font.size': 18}) # must set in top

# data
x = words_find_characterization
y_true_5 = list((df5.apply(pd.Series.value_counts, axis=0)).iloc[1])
y_true_4 = list((df4.apply(pd.Series.value_counts, axis=0)).iloc[1])
y_true_3 = list((df3.apply(pd.Series.value_counts, axis=0)).iloc[1])
y_true_2 = list((df2.apply(pd.Series.value_counts, axis=0)).iloc[1])
# y_true_1 = list((df1.apply(pd.Series.value_counts, axis=0)).iloc[1]) ## chosen below as threshold value...

# using the existing lists from the OP, create the dataframe
df = pd.DataFrame(data={'x=5': y_true_5, 'x=4': y_true_4, 'x=3': y_true_3, 'x=2': y_true_2}, index=x)

# plot bars or kind='barh' for horizontal bars; adjust figsize accordingly
ax = df.plot(kind='bar', rot=0, xlabel='Characterization Technique', ylabel='True Instances', 
             title='Number of True Instances for each Threshold Value', figsize=(16, 12), fontsize =16)

# add some labels
for c in ax.containers:
    # set the bar label
    ax.bar_label(c, fmt='%.0f', label_type='edge')
    
# add a little space at the top of the plot for the annotation
ax.margins(y=0.1)

# move the legend out of the plot
# ax.legend(title='Threshold Value', bbox_to_anchor=(1, 0.5), loc='upper left', frameon=False)

ax.legend(title='Threshold Value', loc='upper right', frameon=False)

## Plot for the case chosen (x=1), for our case x=1 was satisfactory

### Note: The results were manually verified, for the year 2022, where the results saved in "y_label".

In [ ]:
plt.rcParams.update({'font.size': 18}) # must set in top

# data
x = words_find_characterization
y_true_1 = list((df1.apply(pd.Series.value_counts, axis=0)).iloc[1])
y_label = [0, 10, 3, 2, 2, 2]  ### These true values based on manual labelling, specifically for our case.

# using the existing lists from the OP, create the dataframe
df = pd.DataFrame(data={'True Instances': y_true_1, 'Inaccurate Predictions': y_label}, index=x)

# plot bars or kind='barh' for horizontal bars; adjust figsize accordingly
ax = df.plot(kind='bar', rot=0, xlabel='Characterization Technique', ylabel='No. of articles', 
             title='Characterization techniques ', figsize=(16, 12), fontsize =16)

# add some labels
for c in ax.containers:
    # set the bar label
    ax.bar_label(c, fmt='%.0f', label_type='edge')
    
# add a little space at the top of the plot for the annotation
ax.margins(y=0.1)

# move the legend out of the plot
ax.legend(title=' ', loc='upper right', frameon=False)

### Choose threshold value then save the results at that value.... (x=1 for this example, thus df1 is saved)

### (Comment/Uncomment) -- App4 results

In [ ]:
df1_save = df1  ## create new dataframe to add the doi column
df1_save['doi'] =  characterization_count_final['doi'] ## add the doi back before saving the results
df1_save.to_csv(filename_fulltext_threshold, index=True)  ## comment if already saved 

# # Application 5: Extract Experimental Sections

### For the articles that have TG/TGA analysis, because we are interested in carbon deposition (i.e. catalyst weight reduction)

## # Analysis on the output (application 4) results with TGA (for the year 2022) (change)

In [ ]:
# characterization_count_final = pd.read_excel(filename_fulltext_count, index_col=0) ## read application 4 results, if not loaded

characterization_count_final_TGA = characterization_count_final.loc[(characterization_count_final['TG|TGA'] > 0)]   ### keep rows if TGA mentioned once. 58 article based on app4 results ... (change) condition as needed
characterization_count_final_TGA.shape

# # Extracting the text with the important variables (the inputs)

### Find the paragrpahs that contain the significant parameters: (according to research topic and article structure)
1. synthesis paragraph from the experimental section.... may identify it using the word "catalyst" and "synthesis" or "preparation"
2. most reaction information can be found between the introduciton (the first section) and the result section (third section).... 

### Experimental section extraction for articleswith TG/TGA technique:

In [ ]:
DMR_exp_sec = pd.DataFrame(columns=['original_index', 'doi', 'exp_text']) ## dataframe to save all the results from all articles

dict_exp = dict.fromkeys(['original_index', 'doi', 'exp_text']) ## initialize the dictionary

for k, j in enumerate(characterization_count_final_TGA.index): ## loop throught all the articles of interest (in this case has TGA analysis)
    df_papers = pd.read_excel(filename_full_text, index_col=0, sheet_name = j)  ### file name for full texts results
    df_papers_string = ""  ## string to append the seperate sections for each article into one variable, defined for each article loop

    for i in np.arange(1, df_papers.shape[0]-1):  ## begin at 1: to exclude the introduction && -1 is: to exclude the last paragraph, usually the conclusion
#     print(df_papers['sections'][i]) ### for testing
        df_papers_string = df_papers_string + str(df_papers['Text'][i]) 
#         print(df_papers_string)  ### for testing
        if 'result' in df_papers['sections'][i].lower(): ## loop over the sections until the "results" section is found 
#             print("results section found, breaking the for loop.............") ## stop when the results section is found, i.e. extraxt the text from every seciton before the results section
#             print(df_papers['sections'][i])  ### for testing
            if df_papers_string == dict_exp['exp_text']: ### if new string is same as previous text, then the experimental section is not available
                df_papers_string = 'nan'
            ## this line must be before the for loop to save the results for each article, then start the next loop for another article
            dict_exp = {'original_index':j, 'doi':characterization_count_final_TGA.loc[characterization_count_final_TGA.index == j]['doi'].item(), 'exp_text': df_papers_string} ## dictionary to save the experimental section for particular journal article defined by the DOI
            ## note: the doi is taken from the characterization_count_final_TGA dataframe (it has the original index as the main index)
            break

    exp_df = pd.DataFrame(data=dict_exp, index=[j])
    DMR_exp_sec = pd.concat([DMR_exp_sec,exp_df], axis=0, ignore_index=True)

### (Comment/Uncomment) -- App5 results

In [ ]:
## use Excel to save each year result in new sheet ## comment if already saved 

if os.path.isfile(filename_fulltext_exp_sections): ### check if file exist, then add sheet, otherwise make new file
    with pd.ExcelWriter(filename_fulltext_exp_sections, engine = 'openpyxl', mode='a') as writer_app5:
        DMR_exp_sec.to_excel(writer_app5, sheet_name = f"{search_year}", index=True)  ### save the resulted rows, each year in different sheet
else:
    with pd.ExcelWriter(filename_fulltext_exp_sections, engine = 'openpyxl') as writer_app5:
        DMR_exp_sec.to_excel(writer_app5, sheet_name = f"{search_year}", index=True)  ### save the resulted rows, each year in different sheet
# writer_app5.close()

In [ ]:
# DMR_exp_sec = pd.read_csv(filename_fulltext_exp_sections, index_col=0) ## Read if not previously defined
print(DMR_exp_sec.shape)

In [ ]:
pd.options.display.max_colwidth = None
pd.options.display.max_rows = None
display(DMR_exp_sec.head())